In [0]:
WITH monthly_employee_sales AS (
    SELECT 
        e.EmployeeID,
        CONCAT(e.FirstName, ' ', e.LastName) AS EmployeeName,
        CONCAT(YEAR(i.InvoiceDate), '-Q', QUARTER(i.InvoiceDate)) AS YearQuarter,
        CASE 
            WHEN MONTH(i.InvoiceDate) IN (1, 4, 7, 10) THEN 1
            WHEN MONTH(i.InvoiceDate) IN (2, 5, 8, 11) THEN 2
            WHEN MONTH(i.InvoiceDate) IN (3, 6, 9, 12) THEN 3
        END AS MonthNum,
        CASE 
            WHEN MONTH(i.InvoiceDate) IN (1, 4, 7, 10) THEN 'Month 1'
            WHEN MONTH(i.InvoiceDate) IN (2, 5, 8, 11) THEN 'Month 2'
            WHEN MONTH(i.InvoiceDate) IN (3, 6, 9, 12) THEN 'Month 3'
        END AS QuarterMonth,
        SUM(fact.UnitPrice * fact.Quantity) AS Revenue
    FROM silver_invoiceline fact
    LEFT JOIN silver_invoice i ON fact.InvoiceId = i.InvoiceId
    LEFT JOIN silver_customer c ON i.CustomerId = c.CustomerId
    LEFT JOIN silver_employee e ON c.SupportRepId = e.EmployeeId
    GROUP BY 
        e.EmployeeID,
        CONCAT(e.FirstName, ' ', e.LastName),
        CONCAT(YEAR(i.InvoiceDate), '-Q', QUARTER(i.InvoiceDate)),
        MONTH(i.InvoiceDate)
)
SELECT 
    EmployeeID,
    EmployeeName,
    YearQuarter,
    QuarterMonth,
    Revenue,
    AVG(CASE WHEN MonthNum IN (1, 2) THEN Revenue END) OVER (
        PARTITION BY EmployeeID, YearQuarter
    ) AS Avg_Early_Months_Revenue
FROM monthly_employee_sales
ORDER BY 
    YearQuarter, 
    EmployeeID, 
    MonthNum;